# Initial Analysis

Goal: understand the data (columns, data types, missing values, duplicates).

[data-profile.md](../reports/data-profile.md) lists the automated checks; verify its flags here and decide what to do about each one.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df_nightly = pd.read_csv('../data/raw/sleep_nightly.csv')
df_scores = pd.read_csv('../data/raw/sleep_scores.csv')

df = pd.merge(df_nightly, df_scores, how='inner', on='night')

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 417 entries, 0 to 416
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   night             417 non-null    str    
 1   bedtime_x         417 non-null    str    
 2   wake_time         417 non-null    str    
 3   asleep_min        417 non-null    float64
 4   core_min          417 non-null    float64
 5   deep_min          417 non-null    float64
 6   rem_min           417 non-null    float64
 7   awake_min_x       417 non-null    float64
 8   unspecified_min   417 non-null    float64
 9   in_bed_min        417 non-null    int64  
 10  est_score         417 non-null    int64  
 11  category          417 non-null    str    
 12  duration_pts      417 non-null    float64
 13  bedtime_pts       417 non-null    float64
 14  interruption_pts  415 non-null    float64
 15  asleep_h          417 non-null    float64
 16  bedtime_y         417 non-null    str    
 17  bedtime_

## Data Profile Flags

### 1:1 Mappings

With the merging of the two .csv files, some new columns with potential 1:1 mappings appeared that weren't caught by the data profiler.

In [3]:
print(df['night'].equals(df['bedtime_x']))
print(df['night'].equals(df['wake_time']))
print(df['bedtime_x'].equals(df['wake_time']))

df[['night', 'bedtime_x', 'wake_time']].head()

False
False
False


,night,bedtime_x,wake_time
0,2025-09-19,2025-09-19 23:38,2025-09-20 06:49
1,2025-09-20,2025-09-20 21:56,2025-09-21 07:38
2,2025-09-21,2025-09-21 22:00,2025-09-22 06:30
3,2025-09-22,2025-09-22 22:02,2025-09-23 07:00
4,2025-09-23,2025-09-23 22:14,2025-09-24 06:42


The profiler flags all three columns as identical (1:1 mappings), but here we can see that that's not the case. All three columns could be useful for the analysis, therefore neither one will be dropped.

In [4]:
print(df['bedtime_x'].equals(df['bedtime_y']))

print(df[['bedtime_x', 'bedtime_y']].head())

False
          bedtime_x bedtime_y
0  2025-09-19 23:38     23:38
1  2025-09-20 21:56     21:56
2  2025-09-21 22:00     22:00
3  2025-09-22 22:02     22:02
4  2025-09-23 22:14     22:14


In [5]:
print(df['awake_min_x'].equals(df['awake_min_y']))

print(df[['awake_min_x', 'awake_min_y']].head())

True
   awake_min_x  awake_min_y
0         15.4         15.4
1         33.3         33.3
2         43.3         43.3
3         29.4         29.4
4         55.7         55.7


*bedtime_x* gives the date and the time when the sleep cycle began, *bedtime_y* gives only the time. Therefore, *bedtime_y* will be dropped, and *bedtime_x* will be renamed to *bedtime*.

*awake_min_x* and *awake_min_y* are identical. *awake_min_y* will be dropped, and *awake_min_x* will be renamed to *awake_min*.

### Constant Columns

In [6]:
print(df['in_bed_min'].nunique())
print(df['in_bed_min'].unique())

1
[0]


*in_bed_min* can be dropped as it's always 0.

### Outliers

There are many rows flagged with **many IQR outliers**, and outliers in this dataset doesn't mean wrong. Therefore, all of them will be kept.

### Missing

In [7]:
df.isna().sum()

night                 0
bedtime_x             0
wake_time             0
asleep_min            0
core_min              0
deep_min              0
rem_min               0
awake_min_x           0
unspecified_min       0
in_bed_min            0
est_score             0
category              0
duration_pts          0
bedtime_pts           0
interruption_pts      2
asleep_h              0
bedtime_y             0
bedtime_dev_min       3
wake_ups              2
awake_min_y           0
notes               412
dtype: int64

In [8]:
print(f'Notes: {df['notes'].isna().sum()} missing values - {(df['notes'].isna().sum() / df.shape[0]):.1%}')

Notes: 412 missing values - 98.8%


The notes column will be removed, as 98.8% of its values are missing.

In [9]:
same_rows = df['interruption_pts'].isna().equals(df['wake_ups'].isna())

print(same_rows)

df[df['interruption_pts'].isna() | df['wake_ups'].isna()]

True


,night,bedtime_x,wake_time,asleep_min,core_min,deep_min,rem_min,awake_min_x,unspecified_min,in_bed_min,...,category,duration_pts,bedtime_pts,interruption_pts,asleep_h,bedtime_y,bedtime_dev_min,wake_ups,awake_min_y,notes
288,2026-07-13,2026-07-14 01:28,2026-07-14 04:26,152.3,0.0,0.0,0.0,25.4,152.3,0,...,Very Low,0.0,16.2,NaN,2.54,01:28,99.0,NaN,25.4,no stage data: interruptions not scored
351,2026-09-16,2026-09-17 03:21,2026-09-17 07:47,218.1,0.0,0.0,0.0,32.3,218.1,0,...,Very Low,0.0,0.0,NaN,3.63,03:21,214.0,NaN,32.3,no stage data: interruptions not scored


*interruption_pts* and *wake_ups* both have two missing values, and both of them are in the same rows (as the code above suggests). Therefore these two rows will be dropped.

In [10]:
df[df['bedtime_dev_min'].isna()]

,night,bedtime_x,wake_time,asleep_min,core_min,deep_min,rem_min,awake_min_x,unspecified_min,in_bed_min,...,category,duration_pts,bedtime_pts,interruption_pts,asleep_h,bedtime_y,bedtime_dev_min,wake_ups,awake_min_y,notes
0,2025-09-19,2025-09-19 23:38,2025-09-20 06:49,416.3,273.3,39.8,103.1,15.4,0.0,0,...,High,42.0,30.0,10.7,6.94,23:38,NaN,6.0,15.4,too little history: bedtime given full points
1,2025-09-20,2025-09-20 21:56,2025-09-21 07:38,548.2,372.4,53.2,122.5,33.3,0.0,0,...,High,50.0,30.0,6.4,9.14,21:56,NaN,10.0,33.3,too little history: bedtime given full points
2,2025-09-21,2025-09-21 22:00,2025-09-22 06:30,466.5,295.2,58.2,113.0,43.3,0.0,0,...,High,50.0,30.0,4.0,7.78,22:00,NaN,12.0,43.3,too little history: bedtime given full points


*bedtime_dev_min* helps with ranking bedtime, and in the first three rows, there is too little history in order to compare it. These rows won't be dropped, as they're automatically ranked with maximum bedtime points.

## To Do

- Convert *night* (%Y-%m-%d), *bedtime_x* (%Y-%m-%d %H:%M), *wake_time* to dates (%Y-%m-%d %H:%M).
- Drop *bedtime_y*, *awake_min_y*, *notes*.
- Drop rows where *interruption_pts* and *wake_ups* are missing.
- Rename *bedtime_x* to *bedtime*, *awake_min_x* to *awake_min*.
- Convert *wake_ups* to int (it's stored as float because of missing values).